# MODULE 5: ROOT-CAUSE SYNTHESIS & BUSINESS PRIORITIZATION
## From Validated Statistical Evidence to Executive Business Decisions
**Project:** Gradient Learnings Data Analytics Hackathon 2026 — Olist Brazilian E-Commerce Marketplace Diagnostic  
**Lead Analytics & Decision Science Team:** Strategic Analytics & Decision Engineering  
**Dataset Grain:** 1 Row = 1 Order (`order_id`) | Canonical Base: `data/processed/analytical_model.parquet`  
**Standard:** 100% Zero-Trust Audited Statistical Evidence  

---

### Executive Mandate
This notebook transitions from **"What explains performance?"** (Module 4) to:
> **"Where should Olist intervene first to create the largest measurable improvement in customer experience?"**

Every recommendation in this notebook follows the strict chain:
`Evidence → Finding → Affected Segment → Operational Mechanism → Action → KPI`


## 1. Setup & Environment Initialization
We load the validated canonical analytical model and pre-computed Module 5 diagnostic tables.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Global Paths
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
TABLES_DIR = BASE_DIR / "outputs" / "tables"
FIGURES_DIR = BASE_DIR / "outputs" / "figures"

# Style Configuration
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8
COLOR_PRIMARY = '#1E3A8A'
COLOR_SECONDARY = '#0D9488'
COLOR_DANGER = '#DC2626'
COLOR_ACCENT = '#D97706'

# Load Canonical Analytical Base
df = pd.read_parquet(PROCESSED_DIR / "analytical_model.parquet")
is_pop_e = (df['order_status'] == 'delivered') & df['order_delivered_customer_date'].notna() & df['review_score'].notna()
df_e = df[is_pop_e].copy()
df_e['low_review'] = (df_e['review_score'] <= 2).astype(int)

print(f"Loaded Canonical Analytical Model: N = {len(df):,} total orders")
print(f"Population E (Delivered & Reviewed): N = {len(df_e):,} orders | Low Reviews = {df_e['low_review'].sum():,} ({df_e['low_review'].mean()*100:.2f}%)")


## 2. Audited Evidence Base
Module 5 synthesizes the verified findings from Modules 0–4:
* **Finding A (Structural Delay Breakpoint):** Econometric break at $\tau = 0.5\text{d}$ late; escalation threshold at $\tau = 3.5\text{d}$ late.
* **Finding B (Survey Timing Asynchrony):** Pre-delivery surveys exhibit an independent adjusted Odds Ratio of $12.50\text{x}$ (strict calendar) and $19.59\text{x}$ (answered pre-delivery).
* **Finding C (Carrier Bottleneck):** Carrier transit represents $82.5\%$ of fulfillment time and $4.0\text{x}$ the excess odds per standard deviation vs. merchant handling.
* **Finding D (Geographic Mediation):** Distance is mediated by delivery duration.
* **Finding E (Category Exposure):** Category interaction explains $< 0.1\%$ of variance (statistical noise).
* **Finding F (Freight Neutrality):** Freight share has no meaningful direct effect once timeliness is controlled.
* **Finding G (Black Friday Concentration):** November 2017 collapse was concentrated in carrier linehaul transit ($+5.2\text{d}$).


In [ ]:
# Load Root-Cause Contribution Matrix
matrix_df = pd.read_csv(TABLES_DIR / "root_cause_contribution_matrix.csv")
display(matrix_df[['Level', 'Factor', 'Evidence_Type', 'Effect_Size', 'Actionability', 'Priority']])


## 3. Four-Level Root Cause Hierarchy
We synthesize the marketplace dissatisfaction system into 4 distinct evidentiary tiers:
1. **Level 1 — Structural:** Exogenous geography and long-haul trunkline exposure.
2. **Level 2 — Operational:** Carrier linehaul bottlenecks and promised date breaches.
3. **Level 3 — Experience Amplifiers:** Automated CRM feedback timing asynchrony.
4. **Level 4 — Context & Friction:** Merchandise categories, order value, and freight share.


In [ ]:
# Display Visual Root-Cause Hierarchy Table
display(Image(filename=str(FIGURES_DIR / "fig25_root_cause_contribution_matrix.png")))


## 4. Delivery Delay Exposure & Non-Linear Operational Collapse
Delays past promised delivery dates trigger severe satisfaction collapse. Here we analyze the empirical distribution across delay severity strata.


In [ ]:
delay_df = pd.read_csv(TABLES_DIR / "high_risk_delay_segments.csv")
print("=== High-Risk Delivery Delay Strata ===")
display(delay_df)

severe_orders = delay_df.loc[delay_df['delay_stratum'].isin(['Moderate Late (4-7d late)', 'Severe Late (>7d late)']), 'orders'].sum()
severe_low = delay_df.loc[delay_df['delay_stratum'].isin(['Moderate Late (4-7d late)', 'Severe Late (>7d late)']), 'low_reviews'].sum()
print(f"\nDisproportionate Exposure: Orders late >3 days represent {severe_orders:,} orders ({severe_orders/95824*100:.2f}% of volume) but generate {severe_low:,} low reviews ({severe_low/12272*100:.2f}% of all delivered low reviews)!")


## 5. High-Risk Seller Operational Cohorts
We establish explicit sample-size threshold rules ($N \ge 100$ orders) to avoid penalizing small merchants for statistical noise. We identify seller cohorts rather than individual 'bad actors'.


In [ ]:
seller_df = pd.read_csv(TABLES_DIR / "high_risk_seller_segments.csv")
display(seller_df)


## 6. High-Risk Geographic Corridors (SP -> RJ Deep Dive)
Evaluating state-to-state corridors with $N \ge 100$ orders reveals massive concentration of operational friction on the primary São Paulo to Rio de Janeiro linehaul corridor.


In [ ]:
corr_df = pd.read_csv(TABLES_DIR / "high_risk_geographic_segments.csv")
display(corr_df.head(10)[['corridor', 'orders', 'late_rate', 'severe_late_rate', 'low_review_rate', 'dissatisfaction_share_pct', 'median_delay_days', 'median_duration_days']])

display(Image(filename=str(FIGURES_DIR / "fig27_corridor_risk_bubble_scatter.png")))


## 7. Product Category Exposure: Volume Exposure vs. Root Cause
Because category moderation interaction explains only $0.073\%$ of review score variance, category is not a root cause. However, heavy categories generate massive exposure due to sheer order volume.


In [ ]:
cat_df = pd.read_csv(TABLES_DIR / "high_risk_category_segments.csv")
display(cat_df.head(10)[['dominant_category', 'orders', 'low_reviews', 'share_of_all_low_reviews', 'low_review_rate', 'late_rate', 'severe_late_rate']])


## 8. Mutually Exclusive Survey Timing Segments & CRM Amplification
Zero-Trust timestamp forensic analysis established that raw Definition A ($N=8,140$) included $3,164$ same-day deliveries due to midnight date truncation.
Here we inspect the mutually exclusive partition with zero double counting.


In [ ]:
survey_df = pd.read_csv(TABLES_DIR / "survey_timing_segments.csv")
display(survey_df)


## 9. Compound High-Impact Risk Intersections
Searching for the intersection of operational delay, geographic corridors, and feedback timing uncovers acute operational failure zones.


In [ ]:
combo_df = pd.read_csv(TABLES_DIR / "high_impact_combinations.csv")
display(combo_df[['Combination_ID', 'Combination_Name', 'Order_Count', 'Order_Share_Pct', 'Low_Reviews', 'Low_Review_Rate_Pct', 'Dissatisfaction_Share_Pct']])

display(Image(filename=str(FIGURES_DIR / "fig26_high_impact_segment_exposure_matrix.png")))


## 10. Comprehensive Business Exposure Matrix
We calculate Order Exposure, Revenue Exposure, Dissatisfaction Exposure, and Delay Exposure across all target operational segments.


In [ ]:
exp_df = pd.read_csv(TABLES_DIR / "business_exposure_segments.csv")
display(exp_df)


## 11. Prioritization Scoring & P0 / P1 / P2 Operational Framework
We formulate a transparent, four-factor prioritization scoring model:
$$\text{Priority Score} = \text{Severity (1–5)} \times \text{Exposure (1–5)} \times \text{Actionability (1–5)} \times \text{Evidence Confidence (1–5)}$$


In [ ]:
p_df = pd.read_csv(TABLES_DIR / "prioritization_score.csv")
display(p_df[['Intervention_ID', 'Intervention_Name', 'Priority_Score', 'P_Rank', 'Addressable_Low_Reviews', 'Owner']])

display(Image(filename=str(FIGURES_DIR / "fig28_p0_p1_p2_opportunity_matrix.png")))


## 12. Actionable Operational Roadmap (P0 / P1 / P2)
For each priority tier, we specify the Problem, Evidence, Mechanism, Action, KPI, and Target Impact without ungrounded causal promises.


In [ ]:
for _, r in p_df.iterrows():
    print(f"[{r['P_Rank'].upper()}] {r['Intervention_ID']}: {r['Intervention_Name']}")
    print(f"  - Target Segment: {r['Target_Segment']}")
    print(f"  - Priority Score: {r['Priority_Score']} (Sev={r['Severity_Score']}, Exp={r['Exposure_Score']}, Act={r['Actionability_Score']}, Ev={r['Evidence_Score']})")
    print(f"  - Addressable Exposure: {r['Addressable_Low_Reviews']:,} low reviews | R${r['Addressable_GMV_BRL']:,.2f} GMV")
    print(f"  - Operational Impact: {r['Target_Reduction_Impact']}")
    print(f"  - Accountable Owner: {r['Owner']}\n")


## 13. Executive Operational Scorecard & Monitoring Cadence
To ensure governance, we establish explicit thresholds, targets, and monitoring cadences for executive leadership.


In [ ]:
card_df = pd.read_csv(TABLES_DIR / "executive_scorecard.csv")
display(card_df[['KPI_Code', 'Metric_Name', 'Current_Value', 'Target_Benchmark', 'Priority_Tier', 'Accountable_Owner', 'Cadence']])

display(Image(filename=str(FIGURES_DIR / "fig30_executive_prioritization_scorecard.png")))


## 14. Signature Business Insight: The Three-Tiered System
Olist's customer dissatisfaction is not a single point failure; it is an unaligned three-tier system:
1. **Structural Base:** Geographic concentration ($70.3\%$ SP sellers) generates long-haul linehaul exposure.
2. **Operational Driver:** Carrier transit accounts for $82.5\%$ of fulfillment time and $4.0\text{x}$ the excess odds of low reviews.
3. **Amplification Mechanism:** Automated CRM surveys sent while packages are delayed in transit amplify dissatisfaction by $12.5\text{x}$ and account for $26.1\%$ of marketplace low reviews.


In [ ]:
display(Image(filename=str(FIGURES_DIR / "fig29_root_cause_waterfall_decision_framework.png")))


## 15. Module 5 Summary & Formal Finding Register
Module 5 provides the verified, peer-defensible operational roadmap linking data evidence directly to executive decisions.


In [ ]:
find_df = pd.read_csv(TABLES_DIR / "module5_finding_register.csv")
display(find_df[['Finding', 'Metric', 'Exposure', 'Action', 'Priority', 'Confidence']])
